In [ ]:

import os
import math
import requests
import json
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, HTML
from urllib.parse import parse_qs, urlparse
import sys

API_BASE_URL = "http://localhost:3000/api/v1/lakehouse/network-analysis/transaction/"
FALLBACK_ACCOUNT_ID = "dbtrAcct_24a03dafa2c14f6da6bfc195d57c6d21"

def fetch_transaction_network(account_id):
    url = f"{API_BASE_URL}{account_id}"
    try:
        response = requests.get(url)
        response.raise_for_status()
        return response.json()
    except Exception:
       
        return None

def get_account_id_from_url():
    
    if len(sys.argv) > 0 and '?' in sys.argv[0]:
        query = urlparse(sys.argv[0]).query
        params = parse_qs(query)
        return params.get('accountId', [FALLBACK_ACCOUNT_ID])[0]
    return os.getenv('ACCOUNT_ID', FALLBACK_ACCOUNT_ID)


In [ ]:

account_id = get_account_id_from_url()
data = fetch_transaction_network(account_id)


In [ ]:
if not data:
    display(HTML(f"<div style='color:#ef4444'>No network data available for account: {account_id}</div>"))
else:
    
    nodes = []
    edges = []
    
    center = data.get('centerAccount')
    if center:
        nodes.append({
            'id': center.get('accountId'),
            'label': center.get('accountHolder'),
            'type': 'center',
            'status': 'normal', 
            'isCenter': True
        })
        
    for acc in data.get('connectedAccounts', []):
        nodes.append({
            'id': acc.get('accountId'),
            'label': acc.get('accountHolder'),
            'type': 'connected',
            'status': 'alert' if acc.get('hasAlert') else 'normal',
            'isCenter': False
        })
        
    edges = data.get('edges', [])
    
    # Calculate positions
    positions = {}
    center_idx = 0
  
    for i, n in enumerate(nodes):
        if n.get('isCenter'):
            center_idx = i
            break
            
    radius = 250
    n_nodes = len(nodes)
    angle_step = 2 * math.pi / max(n_nodes - 1, 1)
    j = 0
    for i, node in enumerate(nodes):
        if i == center_idx:
            positions[node['id']] = (0.0, 0.0)
        else:
            angle = angle_step * j
            positions[node['id']] = (radius * math.cos(angle), radius * math.sin(angle))
            j += 1

    # Create edge traces with different styles
    solid_edge_x = []
    solid_edge_y = []
    dashed_edge_x = []
    dashed_edge_y = []
    
    center_id = center.get('accountId') if center else None
    
    for e in edges:
        s = e.get('source')
        t = e.get('target')
        if s in positions and t in positions:
            x0, y0 = positions[s]
            x1, y1 = positions[t]
            
            # Outbound from center = solid, Inbound to center = dashed
            if s == center_id:
                solid_edge_x += [x0, x1, None]
                solid_edge_y += [y0, y1, None]
            else:
                dashed_edge_x += [x0, x1, None]
                dashed_edge_y += [y0, y1, None]

    # Solid edges (outbound)
    solid_edge_trace = go.Scatter(
        x=solid_edge_x, y=solid_edge_y, 
        mode='lines', 
        line=dict(width=2, color='#f97316', dash='solid'), 
        hoverinfo='none',
        showlegend=False
    )
    
    # Dashed edges (inbound)
    dashed_edge_trace = go.Scatter(
        x=dashed_edge_x, y=dashed_edge_y, 
        mode='lines', 
        line=dict(width=2, color='#94a3b8', dash='dash'), 
        hoverinfo='none',
        showlegend=False
    )

    node_x = []
    node_y = []
    node_text = []
    marker_size = []
    marker_color = []
    marker_line_color = []
    marker_line_width = []
    node_ids = []

    for n in nodes:
        nid = n.get('id')
        if nid not in positions: continue
        x, y = positions[nid]
        node_x.append(x)
        node_y.append(y)
        label = n.get('label') or nid
        node_text.append(f"<b>{nid}</b><br>{label}")
        
        is_center = n.get('isCenter')
        status = n.get('status')
        
        if is_center:
            color = '#fef3c7'  # Light yellow fill
            line_color = '#f59e0b'  # Orange border
            size = 45
            line_width = 4
        elif status == 'alert':
            color = '#fee2e2'  # Light red fill
            line_color = '#ef4444'  # Red border
            size = 35
            line_width = 3
        else:
            color = '#e0e7ff'  # Light blue fill
            line_color = '#6366f1'  # Blue border
            size = 35
            line_width = 2
            
        marker_color.append(color)
        marker_line_color.append(line_color)
        marker_line_width.append(line_width)
        marker_size.append(size)
        node_ids.append(nid)

    node_trace = go.Scatter(
        x=node_x,
        y=node_y,
        mode='markers+text',
        text=[nid.split('-')[-1] if '-' in nid else nid for nid in node_ids],
        hovertext=node_text,
        hoverinfo='text',
        customdata=node_ids,
        marker=dict(
            showscale=False,
            color=marker_color,
            size=marker_size,
            line=dict(width=marker_line_width, color=marker_line_color)
        ),
        textposition='middle center',
        textfont=dict(size=9, color='#1e293b', family='Arial, sans-serif'),
        showlegend=False
    )

    # Calculate stats for sidebar
    connected_count = len([n for n in nodes if not n.get('isCenter')])
    alert_count = len([n for n in nodes if n.get('status') == 'alert'])
    outbound_count = len([e for e in edges if e.get('source') == center_id])
    inbound_count = len([e for e in edges if e.get('target') == center_id])

    fig = go.Figure(
        data=[dashed_edge_trace, solid_edge_trace, node_trace],
        layout=go.Layout(
            title=None,
            showlegend=False,
            hovermode='closest',
            margin=dict(b=40,l=40,r=40,t=40),
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            height=600,
            paper_bgcolor='#ffffff',
            plot_bgcolor='#ffffff',
        )
    )

    graph_id = 'tx-network-graph'
    graph_html = pio.to_html(fig, full_html=False, config={'displayModeBar': False}, div_id=graph_id)
    
    # Legend HTML
    legend_html = """
    <div style="position: absolute; bottom: 20px; left: 20px; background: white; padding: 16px; border-radius: 8px; box-shadow: 0 2px 8px rgba(0,0,0,0.1); font-family: Arial, sans-serif; font-size: 12px;">
        <div style="font-weight: 600; margin-bottom: 12px; color: #1e293b;">Legend</div>
        <div style="display: flex; align-items: center; margin-bottom: 8px;">
            <div style="width: 16px; height: 16px; border-radius: 50%; background: #fee2e2; border: 2px solid #ef4444; margin-right: 8px;"></div>
            <span style="color: #475569;">Alert Triggered</span>
        </div>
        <div style="display: flex; align-items: center; margin-bottom: 8px;">
            <div style="width: 16px; height: 16px; border-radius: 50%; background: #e0e7ff; border: 2px solid #6366f1; margin-right: 8px;"></div>
            <span style="color: #475569;">Normal Account</span>
        </div>
        <div style="display: flex; align-items: center; margin-bottom: 8px;">
            <div style="width: 16px; height: 16px; border-radius: 50%; background: #fef3c7; border: 3px solid #f59e0b; margin-right: 8px;"></div>
            <span style="color: #475569;">Center Account</span>
        </div>
        <div style="border-top: 1px solid #e2e8f0; margin: 12px 0 8px 0;"></div>
        <div style="display: flex; align-items: center; margin-bottom: 6px;">
            <div style="width: 24px; height: 2px; background: #f97316; margin-right: 8px;"></div>
            <span style="color: #475569;">Outbound Flow</span>
        </div>
        <div style="display: flex; align-items: center;">
            <div style="width: 24px; height: 2px; background: #94a3b8; border-top: 2px dashed #94a3b8; margin-right: 8px;"></div>
            <span style="color: #475569;">Inbound Flow</span>
        </div>
    </div>
    """
    
    # Zoom Controls HTML
    zoom_controls_html = """
    <div style="position: absolute; top: 20px; right: 20px; background: white; padding: 8px; border-radius: 8px; box-shadow: 0 2px 8px rgba(0,0,0,0.1); display: flex; flex-direction: column; gap: 8px;">
        <button id="zoom-in-btn" style="width: 36px; height: 36px; border: none; background: #f8fafc; border-radius: 6px; cursor: pointer; display: flex; align-items: center; justify-content: center; font-size: 18px; color: #1e293b; transition: all 0.2s;" onmouseover="this.style.background='#e2e8f0'" onmouseout="this.style.background='#f8fafc'">
            <svg width="20" height="20" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2">
                <circle cx="11" cy="11" r="8"></circle>
                <line x1="11" y1="8" x2="11" y2="14"></line>
                <line x1="8" y1="11" x2="14" y2="11"></line>
                <line x1="21" y1="21" x2="16.65" y2="16.65"></line>
            </svg>
        </button>
        <button id="zoom-out-btn" style="width: 36px; height: 36px; border: none; background: #f8fafc; border-radius: 6px; cursor: pointer; display: flex; align-items: center; justify-content: center; font-size: 18px; color: #1e293b; transition: all 0.2s;" onmouseover="this.style.background='#e2e8f0'" onmouseout="this.style.background='#f8fafc'">
            <svg width="20" height="20" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2">
                <circle cx="11" cy="11" r="8"></circle>
                <line x1="8" y1="11" x2="14" y2="11"></line>
                <line x1="21" y1="21" x2="16.65" y2="16.65"></line>
            </svg>
        </button>
        <button id="reset-zoom-btn" style="width: 36px; height: 36px; border: none; background: #f8fafc; border-radius: 6px; cursor: pointer; display: flex; align-items: center; justify-content: center; font-size: 18px; color: #1e293b; transition: all 0.2s;" onmouseover="this.style.background='#e2e8f0'" onmouseout="this.style.background='#f8fafc'" title="Reset Zoom">
            <svg width="20" height="20" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2">
                <path d="M3 12a9 9 0 0 1 9-9 9.75 9.75 0 0 1 6.74 2.74L21 8"></path>
                <path d="M21 3v5h-5"></path>
                <path d="M21 12a9 9 0 0 1-9 9 9.75 9.75 0 0 1-6.74-2.74L3 16"></path>
                <path d="M3 21v-5h5"></path>
            </svg>
        </button>
        <button id="focus-node-btn" style="width: 36px; height: 36px; border: none; background: #f8fafc; border-radius: 6px; cursor: pointer; display: flex; align-items: center; justify-content: center; font-size: 18px; color: #1e293b; transition: all 0.2s;" onmouseover="this.style.background='#e2e8f0'" onmouseout="this.style.background='#f8fafc'" title="Focus on Selected Node">
            <svg width="20" height="20" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2">
                <circle cx="12" cy="12" r="3"></circle>
                <circle cx="12" cy="12" r="10"></circle>
                <line x1="12" y1="2" x2="12" y2="4"></line>
                <line x1="12" y1="20" x2="12" y2="22"></line>
                <line x1="2" y1="12" x2="4" y2="12"></line>
                <line x1="20" y1="12" x2="22" y2="12"></line>
            </svg>
        </button>
    </div>
    """
    
    # Sidebar HTML
    sidebar_html = f"""
    <div id="sidebar-panel" style="width: 280px; background: white; padding: 24px; border-radius: 8px; box-shadow: 0 2px 8px rgba(0,0,0,0.1); font-family: Arial, sans-serif;">
        <div style="margin-bottom: 24px;">
            <div id="sidebar-title" style="font-size: 18px; font-weight: 600; color: #1e293b; margin-bottom: 16px;">Center Account</div>
            <div style="margin-bottom: 12px;">
                <div style="font-size: 11px; color: #64748b; text-transform: uppercase; margin-bottom: 4px;">ACCOUNT ID</div>
                <div id="sidebar-account-id" style="font-size: 14px; color: #1e293b; font-weight: 500;">{center.get('accountId', '-') if center else '-'}</div>
            </div>
            <div style="margin-bottom: 12px;">
                <div style="font-size: 11px; color: #64748b; text-transform: uppercase; margin-bottom: 4px;">ACCOUNT HOLDER</div>
                <div id="sidebar-account-holder" style="font-size: 14px; color: #1e293b; font-weight: 600;">{center.get('accountHolder', '-') if center else '-'}</div>
            </div>
            <div id="sidebar-status-container" style="display: none; margin-bottom: 12px;">
                <div style="font-size: 11px; color: #64748b; text-transform: uppercase; margin-bottom: 4px;">STATUS</div>
                <div id="sidebar-status" style="display: inline-block; padding: 4px 12px; border-radius: 12px; font-size: 12px; font-weight: 600;"></div>
            </div>
        </div>
        
        <div style="background: #f8fafc; padding: 16px; border-radius: 8px; margin-bottom: 16px;">
            <div id="sidebar-summary-title" style="font-size: 13px; font-weight: 600; color: #1e293b; margin-bottom: 12px;">NETWORK SUMMARY</div>
            <div id="stat-connected" style="display: flex; justify-content: space-between; margin-bottom: 8px;">
                <span style="color: #64748b; font-size: 13px;">Connected Accounts:</span>
                <span id="stat-connected-value" style="font-weight: 600; color: #1e293b;">{connected_count}</span>
            </div>
            <div style="display: flex; justify-content: space-between; margin-bottom: 8px;">
                <span style="color: #64748b; font-size: 13px;">Outbound Connections:</span>
                <span id="stat-outbound-value" style="font-weight: 600; color: #1e293b;">{outbound_count}</span>
            </div>
            <div style="display: flex; justify-content: space-between; margin-bottom: 8px;">
                <span style="color: #64748b; font-size: 13px;">Inbound Connections:</span>
                <span id="stat-inbound-value" style="font-weight: 600; color: #1e293b;">{inbound_count}</span>
            </div>
            <div id="stat-alerts" style="display: flex; justify-content: space-between;">
                <span style="color: #64748b; font-size: 13px;">Accounts with Alerts:</span>
                <span id="stat-alerts-value" style="font-weight: 600; color: #ef4444;">{alert_count}</span>
            </div>
        </div>
        
        <button id="focus-from-sidebar" style="width: 100%; padding: 10px; background: #3b82f6; color: white; border: none; border-radius: 6px; font-size: 13px; font-weight: 600; cursor: pointer; transition: all 0.2s;" onmouseover="this.style.background='#2563eb'" onmouseout="this.style.background='#3b82f6'">
            Focus on This Account
        </button>
    </div>
    """
    
    # Interactive script
    interactive_script = f"""
    <script>
    (function() {{
        const nodes = {json.dumps(nodes)};
        const edges = {json.dumps(edges)};
        const centerAcct = {json.dumps(center)};
        const positions = {json.dumps(positions)};
        const graphDiv = document.getElementById('{graph_id}');
        
        let selectedNodeId = centerAcct ? centerAcct.accountId : null;
        let currentZoomLevel = 1.0;
        
        function updateSidebar(nodeId) {{
            selectedNodeId = nodeId;
            const node = nodes.find(n => n.id === nodeId);
            if (!node) return;
            
            // Update title
            const title = node.isCenter ? 'Center Account' : 'Connected Account';
            document.getElementById('sidebar-title').innerText = title;
            
            // Update account info
            document.getElementById('sidebar-account-id').innerText = node.id;
            document.getElementById('sidebar-account-holder').innerText = node.label || '-';
            
            // Update status badge for non-center accounts
            const statusContainer = document.getElementById('sidebar-status-container');
            if (!node.isCenter) {{
                statusContainer.style.display = 'block';
                const statusEl = document.getElementById('sidebar-status');
                
                if (node.status === 'alert') {{
                    statusEl.innerText = 'ALERT';
                    statusEl.style.background = '#fee2e2';
                    statusEl.style.color = '#dc2626';
                }} else {{
                    statusEl.innerText = 'NORMAL';
                    statusEl.style.background = '#f0f9ff';
                    statusEl.style.color = '#0369a1';
                }}
            }} else {{
                statusContainer.style.display = 'none';
            }}
            
            // Calculate connections for this node
            let inbound = 0, outbound = 0;
            edges.forEach(e => {{
                if (e.source === nodeId) outbound++;
                if (e.target === nodeId) inbound++;
            }});
            
            // Update summary section
            if (node.isCenter) {{
                // Show network summary for center account
                document.getElementById('sidebar-summary-title').innerText = 'NETWORK SUMMARY';
                document.getElementById('stat-connected').style.display = 'flex';
                document.getElementById('stat-alerts').style.display = 'flex';
                
                const connectedCount = nodes.filter(n => !n.isCenter).length;
                const alertCount = nodes.filter(n => n.status === 'alert').length;
                
                document.getElementById('stat-connected-value').innerText = connectedCount;
                document.getElementById('stat-outbound-value').innerText = outbound;
                document.getElementById('stat-inbound-value').innerText = inbound;
                document.getElementById('stat-alerts-value').innerText = alertCount;
            }} else {{
                // Show connections for individual account
                document.getElementById('sidebar-summary-title').innerText = 'CONNECTIONS';
                document.getElementById('stat-connected').style.display = 'none';
                document.getElementById('stat-alerts').style.display = 'none';
                
                document.getElementById('stat-outbound-value').innerText = outbound;
                document.getElementById('stat-inbound-value').innerText = inbound;
            }}
        }}
        
        function zoomIn() {{
            currentZoomLevel *= 1.2;
            applyZoom();
        }}
        
        function zoomOut() {{
            currentZoomLevel /= 1.2;
            applyZoom();
        }}
        
        function resetZoom() {{
            currentZoomLevel = 1.0;
            applyZoom();
        }}
        
        function applyZoom() {{
            const update = {{
                'xaxis.range': [-350 / currentZoomLevel, 350 / currentZoomLevel],
                'yaxis.range': [-350 / currentZoomLevel, 350 / currentZoomLevel]
            }};
            Plotly.relayout(graphDiv, update);
        }}
        
        function focusOnNode() {{
            if (!selectedNodeId || !positions[selectedNodeId]) return;
            
            const pos = positions[selectedNodeId];
            const zoomLevel = 2.5;
            const range = 150;
            
            const update = {{
                'xaxis.range': [pos[0] - range/zoomLevel, pos[0] + range/zoomLevel],
                'yaxis.range': [pos[1] - range/zoomLevel, pos[1] + range/zoomLevel]
            }};
            
            Plotly.relayout(graphDiv, update);
        }}
        
        // Event listeners
        if (graphDiv) {{
            graphDiv.on('plotly_click', d => {{
                if (d.points && d.points[0] && d.points[0].customdata) {{
                    const nodeId = d.points[0].customdata;
                    updateSidebar(nodeId);
                }}
            }});
        }}
        
        document.getElementById('zoom-in-btn').addEventListener('click', zoomIn);
        document.getElementById('zoom-out-btn').addEventListener('click', zoomOut);
        document.getElementById('reset-zoom-btn').addEventListener('click', resetZoom);
        document.getElementById('focus-node-btn').addEventListener('click', focusOnNode);
        document.getElementById('focus-from-sidebar').addEventListener('click', focusOnNode);
    }})();
    </script>
    """
    
    container_html = f"""
    <div style="display: flex; gap: 20px; background: #f1f5f9; padding: 20px; border-radius: 12px;">
        <div style="flex: 1; background: white; border-radius: 8px; box-shadow: 0 2px 8px rgba(0,0,0,0.1); position: relative;">
            {graph_html}
            {legend_html}
            {zoom_controls_html}
        </div>
        {sidebar_html}
    </div>
    {interactive_script}
    """
    display(HTML(container_html))

In [ ]:

import os
from urllib.parse import parse_qs, urlparse
from IPython.display import display, HTML
import sys

def get_account_id_from_url():
    if 'VOILA_BASE_URL' in os.environ:
        if len(sys.argv) > 0 and '?' in sys.argv[0]:
            query = urlparse(sys.argv[0]).query
            params = parse_qs(query)
            return params.get('accountId', [FALLBACK_ACCOUNT_ID])[0]
   
    return FALLBACK_ACCOUNT_ID

account_id = get_account_id_from_url()
transaction_data = fetch_transaction_network(account_id)



In [ ]:
import networkx as nx
import plotly.graph_objects as go

def build_network_graph(data):
    G = nx.DiGraph()
    
    center = data['centerAccount']
    G.add_node(center['accountId'], label=center['accountHolder'], type='center', hasAlert=False)

    for acc in data.get('connectedAccounts', []):
        G.add_node(acc['accountId'], label=acc['accountHolder'], type='connected', hasAlert=acc['hasAlert'])
  
    for edge in data.get('edges', []):
        G.add_edge(edge['source'], edge['target'], type=edge['type'])
    return G

if transaction_data:
    G = build_network_graph(transaction_data)
else:
    G = None